## Deployment Patterns

How you get a model from development into production — and keep it running safely.
Three core patterns: API deployment, containerisation, and batch deployment.
Getting deployment wrong means downtime, wrong predictions, or no rollback option.

## How it actually works

**Pattern 1 — API Deployment (most common for LLM apps):**
Wrap your pipeline in a REST API using FastAPI.
Deploy to cloud — Azure App Service, AWS Lambda, Azure Container Apps.
Any application calls your endpoint and gets predictions back.
This is how the Claude API works — a model behind a REST endpoint.

**Pattern 2 — Containerisation:**
Package your entire application — code, dependencies, environment — into a Docker container.
Deploy the container to cloud.
Ensures identical behaviour everywhere — your laptop, staging, production.
"Works on my machine" becomes irrelevant. The machine is the container.

**Pattern 3 — Batch Deployment:**
For non-real-time use cases, deploy as a scheduled job.
Runs nightly, processes all inputs, writes outputs to database.
No API endpoint needed. No user waiting in real time.

**Blue/Green Deployment — safe rollout:**
Blue = current production version running.
Green = new version being deployed alongside it.
Switch traffic gradually: 10% → 50% → 100%.
If Green fails — switch all traffic back to Blue instantly.
Old version still running. Rollback takes seconds, not hours.

**Shadow Deployment — safe testing:**
New version receives real production traffic but does not return responses to users.
Outputs are logged and compared against the current production model.
Zero risk to users. Full validation on real data.
Promote to production only when comparison confirms quality.

## Where it breaks or gets dangerous

**The most common mistake:**
Deploy new version. Delete old version. Something breaks. Nothing to roll back to.
Always keep the previous version running until new version is proven stable.
Version everything. Store every model artifact. Never overwrite.

**Environment inconsistency:**
Works perfectly in development. Fails in production.
Different Python versions, different library versions, different environment variables.
Docker containers solve this — the environment is part of the deployment.

**Missing rollback plan:**
Every deployment needs a documented rollback procedure.
Who approves the rollback? How long does it take? What triggers it?
In enterprise AI touching financial data — this is not optional.

**Deployment checklist before go-live:**
1. Model evaluated and meets quality threshold
2. API endpoint tested with production-like data
3. Environment variables secured — no hardcoded keys
4. Logging enabled — every request traced
5. Monitoring alerts configured
6. Rollback procedure documented and tested
7. Previous version still accessible

In [ ]:
# Deployment Patterns — conceptual illustration

class ModelDeployment:
    """Simulates a deployed model version."""
    
    def __init__(self, version, accuracy):
        self.version = version
        self.accuracy = accuracy
        self.traffic_percentage = 0
        self.is_active = False
    
    def predict(self, input_data):
        """Simulate a prediction."""
        return {
            "version": self.version,
            "input": input_data,
            "prediction": f"Result from model v{self.version}",
            "confidence": self.accuracy
        }
    
    def __repr__(self):
        return f"Model(v{self.version}, accuracy={self.accuracy:.0%}, traffic={self.traffic_percentage}%)"

def blue_green_deployment(blue, green):
    """Simulate a blue/green deployment rollout."""
    print(f"Starting Blue/Green Deployment")
    print(f"Blue (current): {blue}")
    print(f"Green (new):    {green}")
    print()
    
    rollout_stages = [10, 50, 100]
    
    for stage in rollout_stages:
        green.traffic_percentage = stage
        blue.traffic_percentage = 100 - stage
        
        print(f"Stage: {stage}% traffic to Green")
        print(f"  Blue:  {blue.traffic_percentage}% traffic")
        print(f"  Green: {green.traffic_percentage}% traffic")
        
        # Simulate quality check
        if green.accuracy >= 0.85:
            print(f"  Quality check: ✅ PASSED ({green.accuracy:.0%})")
        else:
            print(f"  Quality check: ❌ FAILED — rolling back to Blue")
            green.traffic_percentage = 0
            blue.traffic_percentage = 100
            print(f"  Rollback complete. Blue serving 100% traffic.")
            return False
        print()
    
    print("✅ Deployment complete. Green is now production.")
    return True

def shadow_deployment(production, shadow, test_inputs):
    """Simulate shadow deployment — compare outputs without affecting users."""
    print("Shadow Deployment — comparing production vs new model")
    print(f"Production: {production}")
    print(f"Shadow:     {shadow}")
    print()
    
    agreements = 0
    for input_data in test_inputs:
        prod_result = production.predict(input_data)
        shadow_result = shadow.predict(input_data)
        match = prod_result["confidence"] == shadow_result["confidence"]
        if match:
            agreements += 1
        print(f"Input: {input_data}")
        print(f"  Production: {prod_result['prediction']} ({prod_result['confidence']:.0%})")
        print(f"  Shadow:     {shadow_result['prediction']} ({shadow_result['confidence']:.0%})")
        print()
    
    print(f"Agreement rate: {agreements}/{len(test_inputs)}")
    print("Users saw zero impact. Shadow ran silently alongside production.")

# Blue/Green example
print("=" * 55)
print("BLUE/GREEN DEPLOYMENT")
print("=" * 55)
blue = ModelDeployment(version="1.2", accuracy=0.91)
green = ModelDeployment(version="1.3", accuracy=0.94)
blue.traffic_percentage = 100
blue_green_deployment(blue, green)

print()
print("=" * 55)
print("SHADOW DEPLOYMENT")
print("=" * 55)
production = ModelDeployment(version="1.2", accuracy=0.91)
shadow = ModelDeployment(version="1.3", accuracy=0.94)
test_inputs = ["Invoice #1001", "Invoice #1002", "Invoice #1003"]
shadow_deployment(production, shadow, test_inputs)

## My D365 analogy

Blue/Green deployment is the technical equivalent of
a D365 parallel run during go-live.

Old system and new system run simultaneously.
Real transactions processed by both.
Results compared.
Only when the new system proves itself — old system is decommissioned.

Every major D365 implementation has a parallel run phase.
You never switch off the old system on day one without verification.
Blue/Green deployment enforces this discipline automatically.

Shadow deployment maps to UAT with production data cloning.
You test the new configuration against real transaction patterns
without affecting the live environment.
Only promote when testing confirms readiness.

The deployment discipline every D365 consultant practises at go-live
is exactly the discipline production AI deployments require.

## LinkedIn post idea

Every D365 go-live has a parallel run phase.
Old system and new system running simultaneously.
Real transactions. Real comparison. Real confidence before cutover.

Nobody switches off the old system on day one.
That would be reckless.

Yet most AI deployments skip this entirely.
New model deployed. Old model deleted. Something breaks. No rollback.

In AI engineering this is called Blue/Green deployment.
Run old and new versions simultaneously.
Shift traffic gradually.
Roll back in seconds if something goes wrong.

14 years of go-live experience taught me one thing:
the systems that fail are the ones with no rollback plan.

That lesson does not change when the system is an AI model.